In [0]:
#Load Data
from pyspark.sql.functions import *

telemetry = spark.table(
    "workspace.predictive_maintenance.silver_telemetry"
)

failures = spark.table(
    "workspace.predictive_maintenance.silver_failures"
)

In [0]:
#Create Failure Flag
failure_flag = failures.select(
    "machineID"
).distinct().withColumn(
    "failure_flag",
    lit(1)
)

display(failure_flag)

machineID,failure_flag
37,1
27,1
71,1
100,1
88,1
80,1
94,1
47,1
12,1
46,1


In [0]:
#Create Machine-Level Features
machine_features = telemetry.groupBy(
    "machineID"
).agg(
    avg("volt").alias("avg_volt"),
    avg("rotate").alias("avg_rotate"),
    avg("pressure").alias("avg_pressure"),
    avg("vibration").alias("avg_vibration")
)

display(machine_features)

machineID,avg_volt,avg_rotate,avg_pressure,avg_vibration
1,170.83389762467777,446.3365017291676,100.66830588816336,40.5863088734261
2,170.76048201063625,446.38891465251436,100.53992979635984,40.301538634559726
3,170.6592348040541,446.5832379901177,100.65311416397488,40.47537610412395
4,170.6597277050074,446.0538013768685,100.56990183131273,40.34161964770795
5,171.04443470435896,446.8585765042036,101.06618841696498,40.264845827522414
6,170.41171548820324,448.5132691907918,100.55627717638285,40.380021869144485
7,171.1800760927571,446.09459516049526,100.82658081265689,40.4901243964111
8,170.59621853024865,447.02614828161074,100.58832750315484,40.367218838073725
9,170.69206861089256,445.7249744682517,100.77190261281486,40.27650668528428
10,170.64417445035767,446.0354167393137,100.50395234193846,40.27462776712498


In [0]:
#Join Failure Information
failure_model_df = machine_features.join(
    failure_flag,
    "machineID",
    "left"
).fillna(0)

display(failure_model_df)

machineID,avg_volt,avg_rotate,avg_pressure,avg_vibration,failure_flag
34,170.9048208577788,447.4467304983512,100.51710354500632,40.27782244589459,1
37,170.55459909641792,445.53931460904994,101.24704758636787,40.69479044693231,1
47,170.9726347097134,445.34503972751355,100.8469650015395,40.214861321961834,1
79,170.86596247470044,446.49103985096576,100.64750419405841,40.435518722302604,1
24,170.68119558805935,447.01802693933473,101.11024891456462,40.4414669787482,1
90,170.9921268232847,445.34179454205616,101.31720147905374,40.28118657723847,1
58,170.71075432663156,445.80276585707264,101.20405721584818,40.25956725435997,1
12,170.9371764931595,447.2425567224375,100.5721741211863,40.295779585232815,1
69,170.43275693741577,446.11516546546954,101.09881777111772,40.50902203216637,1
83,170.90485976018903,445.46782046679965,100.81762563445692,40.638414264591006,1


In [0]:
#Risk Score Calculation
failure_risk = failure_model_df.withColumn(
    "risk_score",
    (
        col("avg_vibration") * 0.6
        +
        col("avg_pressure") * 0.4
    )
)

display(failure_risk)

machineID,avg_volt,avg_rotate,avg_pressure,avg_vibration,failure_flag,risk_score
34,170.9048208577788,447.4467304983512,100.51710354500632,40.27782244589459,1,64.37353488553929
37,170.55459909641792,445.53931460904994,101.24704758636787,40.69479044693231,1,64.91569330270654
47,170.9726347097134,445.34503972751355,100.8469650015395,40.214861321961834,1,64.46770279379291
79,170.86596247470044,446.49103985096576,100.64750419405841,40.435518722302604,1,64.52031291100494
24,170.68119558805935,447.01802693933473,101.11024891456462,40.4414669787482,1,64.70897975307477
90,170.9921268232847,445.34179454205616,101.31720147905374,40.28118657723847,1,64.69559253796459
58,170.71075432663156,445.80276585707264,101.20405721584818,40.25956725435997,1,64.63736323895526
12,170.9371764931595,447.2425567224375,100.5721741211863,40.295779585232815,1,64.40633739961422
69,170.43275693741577,446.11516546546954,101.09881777111772,40.50902203216637,1,64.74494032774692
83,170.90485976018903,445.46782046679965,100.81762563445692,40.638414264591006,1,64.71009881253738


In [0]:
#Top High-Risk Machines
high_risk_machines = failure_risk.orderBy(
    col("risk_score").desc()
)

display(
    high_risk_machines.limit(10)
)

machineID,avg_volt,avg_rotate,avg_pressure,avg_vibration,failure_flag,risk_score
98,170.78247826672967,446.06697430626855,101.70549772443894,40.49960830476827,1,64.98196407263654
99,170.72788359486455,444.7664879806162,101.56885226044444,40.52472953266788,1,64.9423786237785
37,170.55459909641792,445.53931460904994,101.24704758636787,40.69479044693231,1,64.91569330270654
22,170.72227540045975,445.5194430567094,101.50622780531138,40.51131920660993,1,64.90928264609052
73,171.00340953955873,448.41060119714626,101.50409073143614,40.50545545340393,1,64.90490956461682
17,170.726236739353,445.96852475599985,101.34290760938606,40.54904458332405,1,64.86658979374886
20,170.95206335631127,446.56120259853805,101.3084514033588,40.56736476306904,1,64.86379941918494
71,170.90093009515783,446.55202987326624,101.30413812360636,40.48126782558693,1,64.8104159447947
49,170.63774625261178,445.7291044793529,101.0803957777315,40.61879179799484,1,64.8034333898895
87,170.5890863318285,447.0082307567705,101.50063798210319,40.33401473216856,1,64.80066403214241


In [0]:
display(
    failure_risk.select(
        "machineID",
        "risk_score"
    )
)

machineID,risk_score
1,64.619107679321
2,64.39689509927977
3,64.54647132806433
4,64.43293252114987
5,64.58538286329944
6,64.45052399203983
7,64.62470696290941
8,64.45566230410617
9,64.47466505629652
10,64.36635759705038


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    failure_risk.groupBy(
        "failure_flag"
    ).count()
)

failure_flag,count
1,98
0,2


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    high_risk_machines.limit(20)
)

machineID,avg_volt,avg_rotate,avg_pressure,avg_vibration,failure_flag,risk_score
98,170.78247826672967,446.06697430626855,101.70549772443894,40.49960830476827,1,64.98196407263654
99,170.72788359486455,444.7664879806162,101.56885226044444,40.52472953266788,1,64.9423786237785
37,170.55459909641792,445.53931460904994,101.24704758636787,40.69479044693231,1,64.91569330270654
22,170.72227540045975,445.5194430567094,101.50622780531138,40.51131920660993,1,64.90928264609052
73,171.00340953955873,448.41060119714626,101.50409073143614,40.50545545340393,1,64.90490956461682
17,170.726236739353,445.96852475599985,101.34290760938606,40.54904458332405,1,64.86658979374886
20,170.95206335631127,446.56120259853805,101.3084514033588,40.56736476306904,1,64.86379941918494
71,170.90093009515783,446.55202987326624,101.30413812360636,40.48126782558693,1,64.8104159447947
49,170.63774625261178,445.7291044793529,101.0803957777315,40.61879179799484,1,64.8034333898895
87,170.5890863318285,447.0082307567705,101.50063798210319,40.33401473216856,1,64.80066403214241


Databricks visualization. Run in Databricks to view.